# Donut Passport Parser — Kaggle Training Notebook (Fast Version)
This notebook is optimized for **speed and efficiency**.

### Setup Instructions:
1. **Accelerator**: Select GPU T4 x2 or P100.
2. **Internet**: Set to **On** in the sidebar.
3. **Data**: Point to your dataset at `/kaggle/input/datasets/manmohanmehra/donut-training`.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)


In [ ]:
# 1. Install dependencies
!pip install -q sentencepiece transformers datasets albumentations wandb

In [ ]:
# 2. Create directory structure
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/src", exist_ok=True)

In [ ]:
%%writefile /kaggle/working/src/dataset.py
from torch.utils.data import Dataset
from transformers import DonutProcessor
from PIL import Image
import json, random
from pathlib import Path

class PassportDataset(Dataset):
    def __init__(self, data_dir, processor, split="train", max_length=512, val_split=0.1):
        self.processor = processor
        self.max_length = max_length
        self.data_dir = Path(data_dir)

        with open(self.data_dir / "metadata.jsonl") as f:
            records = [json.loads(line) for line in f]

        random.seed(42)
        random.shuffle(records)
        split_idx = int(len(records) * (1 - val_split))
        self.records = records[:split_idx] if split == "train" else records[split_idx:]
        print(f"[{split}] {len(self.records)} samples loaded")

    def __len__(self):
        return len(self.records)

    def _gt_to_token_sequence(self, gt: dict) -> str:
        gt = gt.copy()
        card_type = gt.pop("card_type")
        seq = f"<s_{card_type}>"
        for key, value in gt.items():
            seq += f"<s_{key}>{value or ''}</s_{key}>"
        seq += f"</s_{card_type}>"
        return seq

    def __getitem__(self, idx):
        record = self.records[idx]
        img = Image.open(self.data_dir / "images" / record["file_name"]).convert("RGB")
        pixel_values = self.processor(img, return_tensors="pt").pixel_values.squeeze()
        target_seq = self._gt_to_token_sequence(record["ground_truth"])
        labels = self.processor.tokenizer(
            target_seq, add_special_tokens=False, max_length=self.max_length,
            padding="max_length", truncation=True, return_tensors="pt"
        ).input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
%%writefile /kaggle/working/src/train.py
import os
import sys
import torch
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
sys.path.append("/kaggle/working/src")
from dataset import PassportDataset

# Force single GPU stability
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 1. Paths
INPUT_DIR = "/kaggle/input/datasets/manmohanmehra/donut-training"
dataset_path = os.path.join(INPUT_DIR, "data/augmented")
processor_path = os.path.join(INPUT_DIR, "checkpoints/donut-passport-processor")

# 2. Load processor and model
processor = DonutProcessor.from_pretrained(processor_path, local_files_only=True, use_fast=False)
processor.image_processor.size = {"height": 1280, "width": 960}

model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
model.config.tie_word_embeddings = False
model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(["<s_indian_passport>"])[0]
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

train_dataset = PassportDataset(dataset_path, processor, split="train")
val_dataset   = PassportDataset(dataset_path, processor, split="val")

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/checkpoints/donut-passport-finetuned",
    num_train_epochs=15,             # Focused training
    per_device_train_batch_size=1,   
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    fp16=True,                       
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,     # Revert to best weights at the end
    metric_for_best_model="eval_loss",
    save_total_limit=1,              
    logging_steps=10,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Stop early if no improvement
)

print("🚀 Starting fast training on T4...")
trainer.train()
trainer.save_model("/kaggle/working/checkpoints/donut-passport-final")
processor.save_pretrained("/kaggle/working/checkpoints/donut-passport-final")
print("✅ Training complete!")

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

!python /kaggle/working/src/train.py